In [19]:
from dotenv import load_dotenv
import os
from anthropic import Anthropic

# Load environment variables and initialize Anthropic client
load_dotenv()
client = Anthropic()
model = "claude-haiku-4-5-20251001"

In [20]:
def add_user_message(conversation, content):
    """Appends a user message to the conversation history."""
    conversation.append({
        "role": "user",
        "content": content
    })

def add_assistant_message(conversation, content):
    """Appends an assistant message to the conversation history, cleaning up formatting."""
    cleaned_content = content.replace("**", "").replace("##", " ")
    conversation.append({
        "role": "assistant",
        "content": cleaned_content
    })

def chat(conversation, system=None, stop_sequences=None, stream=False):
    """
    Sends conversation history to Claude.
    Returns the Message response object if stream=False, or accumulated text if stream=True.
    """
    params = {
        "model": model,
        "max_tokens": 500,
        "messages": conversation
    }
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences
        
    if stream:
        full_response = ""
        with client.messages.stream(**params) as stream_obj:
            for text in stream_obj.text_stream:
                print(text, end="", flush=True)
                full_response += text
        print()
        add_assistant_message(conversation, full_response)
        return full_response
    else:
        response = client.messages.create(**params)
        add_assistant_message(conversation, response.content[0].text)
        return response

def ask_ai(question, system=None, stop_sequences=None, stream=False):
    """Helper to add user message and fetch AI response."""
    add_user_message(conversation, question)
    return chat(conversation, system=system, stop_sequences=stop_sequences, stream=stream)

In [21]:
# # --- Demo Conversation ---
# conversation = []
# response = ask_ai("tell me about death note anime?", stream=True)
# response2 = ask_ai("who is the main character?", stream=True)

In [22]:
conversation = []

prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

add_user_message(conversation, prompt)
add_assistant_message(conversation, "```json")

chat_response = chat(conversation, stop_sequences=["```"])

print(chat_response.content[0].text)


[
    {
        "task": "Write a Python function that parses an AWS S3 bucket URI (e.g., 's3://my-bucket/path/to/file.txt') and returns a dictionary with keys 'bucket' and 'key'."
    },
    {
        "task": "Create a regular expression that matches valid AWS IAM role ARNs in the format 'arn:aws:iam::ACCOUNT_ID:role/ROLE_NAME'."
    },
    {
        "task": "Write a Python function that takes an AWS CloudWatch log timestamp in ISO 8601 format and converts it to Unix epoch time in milliseconds."
    }
]



In [23]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""
    
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [ ]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    # TODO - Grading
    score = 10
    
    return {
        "output": output,
        "test_case": test_case,
        "score": score
    }

In [ ]:

dataset = [
    {
        "task": "Write a Python function that parses an AWS S3 bucket URI (e.g., 's3://my-bucket/path/to/file.txt') and returns a dictionary with keys 'bucket' and 'key'."
    },
    {
        "task": "Create a regular expression that matches valid AWS IAM role ARNs in the format 'arn:aws:iam::ACCOUNT_ID:role/ROLE_NAME'."
    },
    {
        "task": "Write a Python function that takes an AWS CloudWatch log timestamp in ISO 8601 format and converts it to Unix epoch time in milliseconds."
    }
]

def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    return results

result = run_eval(dataset)

print(result)

[{'output': Message(id='msg_0145qTPH8owYC2gaPUzzJrV6', container=None, content=[TextBlock(citations=None, text='# AWS S3 URI Parser\n\nHere\'s a Python function that parses S3 bucket URIs:\n\n```python\ndef parse_s3_uri(uri: str) -> dict:\n    """\n    Parse an AWS S3 bucket URI and return bucket name and key.\n    \n    Args:\n        uri: S3 URI in format \'s3://bucket-name/path/to/file\'\n        \n    Returns:\n        Dictionary with \'bucket\' and \'key\' keys\n        \n    Raises:\n        ValueError: If URI format is invalid\n        \n    Example:\n        >>> parse_s3_uri(\'s3://my-bucket/path/to/file.txt\')\n        {\'bucket\': \'my-bucket\', \'key\': \'path/to/file.txt\'}\n    """\n    if not isinstance(uri, str):\n        raise ValueError("URI must be a string")\n    \n    if not uri.startswith(\'s3://\'):\n        raise ValueError("URI must start with \'s3://\'")\n    \n    # Remove the \'s3://\' prefix\n    uri_without_scheme = uri[5:]\n    \n    # Split on the first \

In [32]:
print("Evaluation completed. Results:")
for res in result:
    print("-" * 40)
    print(f"task number {result.index(res) + 1}")
    print(f"Task: {res['test_case']['task']}")
    print(f"Output: {res['output'].content[0].text}")
    print(f"Score: {res['score']}")
    print("-" * 40)


Evaluation completed. Results:
----------------------------------------
task number 1
Task: Write a Python function that parses an AWS S3 bucket URI (e.g., 's3://my-bucket/path/to/file.txt') and returns a dictionary with keys 'bucket' and 'key'.
Output: # AWS S3 URI Parser

Here's a Python function that parses S3 bucket URIs:

```python
def parse_s3_uri(uri: str) -> dict:
    """
    Parse an AWS S3 bucket URI and return bucket name and key.
    
    Args:
        uri: S3 URI in format 's3://bucket-name/path/to/file'
        
    Returns:
        Dictionary with 'bucket' and 'key' keys
        
    Raises:
        ValueError: If URI format is invalid
        
    Example:
        >>> parse_s3_uri('s3://my-bucket/path/to/file.txt')
        {'bucket': 'my-bucket', 'key': 'path/to/file.txt'}
    """
    if not isinstance(uri, str):
        raise ValueError("URI must be a string")
    
    if not uri.startswith('s3://'):
        raise ValueError("URI must start with 's3://'")
    
    # Re